In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
import numpy as np, pandas as pd, os, json, re, socket, tempfile, threading, time
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
pip install fastapi uvicorn pyngrok transformers==4.52.4 accelerate -q

In [ ]:
pip -q install "transformers>=4.43" "accelerate>=0.33" "bitsandbytes>=0.43" "autoawq>=0.2.7" torch --extra-index-url https://download.pytorch.org/whl/cu121

In [ ]:
pip install langchain langchain-community langchain-core transformers==4.52.4

In [ ]:
!pip install sentence-transformers PyPDF2 faiss-cpu

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain.llms.base import LLM
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from sentence_transformers import SentenceTransformer
import faiss
from langchain.document_loaders import PyPDFLoader

In [ ]:
base_id = "Qwen/Qwen2.5-7B-Instruct"
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(base_id, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(base_id, quantization_config=bnb_cfg, device_map="auto").eval()
print("Model ready")

In [ ]:
def generate_text(prompt, max_new_tokens=2048):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        pad_token_id=tokenizer.eos_token_id,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

class CustomHFLLM(LLM):
    def _call(self, prompt: str, stop=None) -> str:
        return generate_text(prompt, max_new_tokens=2048)
    @property
    def _llm_type(self):
        return "custom_qwen"

llm = CustomHFLLM()

In [ ]:
response_schemas = [
    ResponseSchema(
        name="technical_questions",
        description="List of technical interview questions. Each item must include: question, suggested_answer (short), and relevance_score  (0-1)."
    ),
    ResponseSchema(
        name="behavioral_questions",
        description="List of behavioral interview questions. Each item must include: question, suggested_answer (short), and relevance_score  (0-1)."
    )
]

parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = parser.get_format_instructions()

In [ ]:
question_prompt = PromptTemplate(
    input_variables=["resume", "job_description", "format_instructions"],
    template="""
You are an expert interview coach.

Given the candidate resume:
---
{resume}
---

And the job description:
---
{job_description}
---

Produce a mock interview JSON with:
- technical_questions: array of [question, suggested_answer, relevance_score]
- behavioral_questions: array of [question, suggested_answer, relevance_score]

Notes:
- provide 6-7 technical questions
- provide 3-5 behavioral questions
- Prioritize role-specific technical topics from the job description.
- Suggested answers should be concise (1-3 sentences).
- relevance_score should be the model's estimate (0.0 - 1.0) of how well the candidate can answer.
Return ONLY the JSON.

{format_instructions}
"""
)

In [ ]:
from typing import List
EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMB_MODEL_NAME)

def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> List[str]:
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        ch = " ".join(words[i:i+chunk_size])
        if ch.strip():
            chunks.append(ch)
    return chunks

def embed_chunks(chunks, model_name='sentence-transformers/all-MiniLM-L6-v2'):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, convert_to_numpy=True)
    return model, embeddings

def create_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index

def search_index(query, model, index, chunks, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)
    return distances[0], [chunks[i] for i in indices[0]]

def compute_relevance(question, embed_model, faiss_index, resume_chunks):
    distances, retrieved = search_index(
        question, embed_model, faiss_index, resume_chunks, k=3
    )
    avg_dist = float(distances.mean())
    score = 1 / (1 + avg_dist)
    return min(max(score, 0.0), 1.0)

In [ ]:
# def load_pdf_text(pdf_bytes):
#     with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
#         tmp.write(pdf_bytes)
#         tmp_path = tmp.name
#     loader = PyPDFLoader(tmp_path)
#     docs = loader.load()
#     text = "\n".join([d.page_content for d in docs])
#     os.remove(tmp_path)
#     return text

In [ ]:
from langchain.document_loaders import PyPDFLoader

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok, conf
import uvicorn
import threading
import socket

NGROK_TOKEN = "35m3XVESAPBT1HgdoBeiDAIf9Vc_3Huddn6vk9dcymBjwSQXv"
API_KEY = "secret123"   # Simple bearer token

app = FastAPI()
# app.add_middleware(
#     CORSMiddleware,
#     allow_origins=["*"],
#     allow_methods=["*"],
#     allow_headers=["*"],
# )

@app.post("/interview")
async def interview_endpoint(
    file: UploadFile = File(...),
    job_description: str = Form(...),
    api_key: str = Form(...)
):
    if api_key != API_KEY:
        raise HTTPException(401, "Unauthorized")

    # Save PDF temporarily
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        tmp.write(await file.read())
        tmp_path = tmp.name

    # Extract text using PyPDFLoader
    loader = PyPDFLoader(tmp_path)
    pages = loader.load()
    resume_text = "\n".join([p.page_content for p in pages])
    chunks = chunk_text(resume_text)
    if not chunks:
        raise HTTPException(400, "Resume is empty")

    embed_model, embeddings = embed_chunks(chunks)
    faiss_index = create_faiss_index(embeddings)


    raw = chain.run(
        resume=resume_text,
        job_description=job_description,
        format_instructions=format_instructions
    )
    try:
        parsed = parser.parse(raw)
    except Exception as e:
        raise HTTPException(500, f"LLM parsing failed: {e}")

    def fill_scores(items):
        out = []
        for item in items:
            question = item.get("question", "")
            answer = item.get("suggested_answer", "")
            score = item.get("relevance_score")
            if score is None:
                score = compute_relevance(question, embed_model, faiss_index, chunks)

            out.append({
                "question": question,
                "suggested_answer": answer,
                "relevance_score": round(float(score), 3)
            })
        return out

    return {
        "technical_questions": fill_scores(parsed.get("technical_questions", [])),
        "behavioral_questions": fill_scores(parsed.get("behavioral_questions", []))
    }

In [ ]:
def find_free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = find_free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("✅ Public URL:", public_url)

def launch():
    uvicorn.run(app, host="0.0.0.0", port=port)

threading.Thread(target=launch, daemon=True).start()
time.sleep(1)